# E005 — CEM Best-Response Search

**Input checklist**
- Required: V2 suite; recommended `macro_library.json` and `archetype_profiles.parquet`.
- Accelerator: **None / CPU**.
- Internet: **OFF**.
- Runtime: scale population/seeds to available CPU. Start small for smoke tests, then increase.

**Why CEM now:** the environment has limited randomness and community reports suggest open-loop strategies are strong. Search a low-dimensional macro policy, not a 720-turn action sequence.

In [ ]:
from pathlib import Path
import os,sys,json
SUITE_CANDIDATES=[Path('/kaggle/input/kaggriculture-v2-suite'),Path('/kaggle/input/kaggriculture-v2-suite/kaggriculture_v2_suite'),Path.cwd().parent,Path.cwd()]
ROOT=next((p for p in SUITE_CANDIDATES if (p/'src'/'kagv2').exists()),None)
if ROOT is None: raise FileNotFoundError('Attach/upload kaggriculture_v2_suite as a Kaggle Dataset, or run this notebook inside the repo.')
sys.path.insert(0,str(ROOT)); WORK=Path('/kaggle/working/kagv2') if Path('/kaggle/working').exists() else ROOT/'artifacts'; WORK.mkdir(parents=True,exist_ok=True)
print('ROOT=',ROOT,'WORK=',WORK)

In [ ]:
import sys,json,numpy as np,pandas as pd
sys.path.insert(0,str(ROOT/'submission'))
from src.kagv2.cem import cem_optimize,save_search
from src.kagv2.simulator import Game
from submission.parametric_agent import ParametricMind,DEFAULT_PARAMS
from submission.base_controller import HarvestMind
from baselines.v1.counter_agent import CounterMeta,TournamentMind
OPPONENT_FACTORIES=[HarvestMind,CounterMeta,TournamentMind]
SEEDS=list(range(4))

In [ ]:
def evaluate(params):
    wins=0.;games=0;margin=0.
    for Of in OPPONENT_FACTORIES:
        for seed in SEEDS:
            for seat in [0,1]:
                mine=ParametricMind(params).act;opp=Of().act;agents=[mine,opp] if seat==0 else [opp,mine]
                s=Game(seed=10000*seed+seat).run(agents);a,b=(s[0],s[1]) if seat==0 else (s[1],s[0]);wins+=1 if a>b else .5 if a==b else 0;margin+=a-b;games+=1
    return wins/games + .01*np.tanh((margin/games)/10000.)
best,hist=cem_optimize(evaluate,iterations=3,population=16,elite_frac=.25,callback=print)
print('BEST',best);save_search(WORK/'cem_best.json',best,hist)

### Serious-run settings
After smoke testing, use ~48–96 candidates/iteration, 8–15 iterations, 16+ seeds, both seats, and a replay-derived opponent zoo. A faster C++ mirror can replace `Game` behind the same evaluator. The optimization target must remain pairwise win rate, not mean coins.